<a href="https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wilsonmarundaa-netizen/updated-flyRank-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logistic regression cause we are trying to predict between two values if the trend is up or down. If its up then it suggests that the page needs monitoring and proctection if down  then it suggests the page needs refreshing or prunning.

In [ ]:
!wget https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv

import pandas as pd

df = pd.read_csv('content_refresh_anonymized.csv')

df["down_trend"] = (df["trend_direction"] == "down").astype(int)

df["down_trend"].head()

df = df.dropna()

df.info()

--2026-08-19 20:34:42--  https://raw.githubusercontent.com/wilsonmarundaa-netizen/updated-flyRank-intern/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘content_refresh_anonymized.csv.6’

content_refresh_ano 100%[===================>]   6.42M  --.-KB/s    in 0.09s   

2026-08-19 20:34:42 (71.3 MB/s) - ‘content_refresh_anonymized.csv.6’ saved [6727670/6727670]

<class 'pandas.core.frame.DataFrame'>
Index: 7350 entries, 6 to 29999
Data columns (total 45 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              7350 non-null   object 
 1   client_id               7350 non-null   object 
 2   

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The column "client_id" represents one row and there is no multiple rows for the same "content_id" then a normal "train_test_split" would do the job.

In [ ]:
from sklearn.model_selection import train_test_split

features =['impressions_90d',
       'clicks_90d', 'pageviews_90d','cpc','sessions_90d', 'users_90d','impressions_last_30d', 'engagement_rate','search_volume']

X = df[features]
y = df["down_trend"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


y.shape
X.shape



(7350, 9)

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

In [ ]:
test_results = X_test.copy()
test_results["model_predictions"] = model.predict(X_test)
test_results["actual_values"] = y_test

import numpy as np

#baseline prediction

test_results["baseline_prediction"] = np.where(
    test_results["impressions_last_30d"] / 30 > test_results["impressions_90d"] / 90,
    0,  #up
    1  #down
)

test_results

,impressions_90d,clicks_90d,pageviews_90d,cpc,sessions_90d,users_90d,impressions_last_30d,engagement_rate,search_volume,model_predictions,actual_values,baseline_prediction
1145,23606,33,72,0.00,72,69,9253,1.39,0.0,0,0,0
12290,5,0,4,0.00,4,4,3,0.00,10.0,1,0,0
15581,1149,0,1,0.00,1,1,21,0.00,0.0,1,1,1
5884,4959,2,14,0.00,7,7,829,0.00,0.0,1,1,1
23740,9215,40,44,0.00,37,37,2910,0.00,0.0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...
22371,421,2,3,0.91,4,4,157,0.00,70.0,1,1,0
11637,1032,1,3,4.48,3,3,281,0.00,210.0,1,0,1
25732,1679,1,4,0.00,4,4,473,0.00,0.0,1,0,1
21761,9971,34,25,0.00,24,24,2088,4.17,0.0,1,1,1


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Precision: Of all the predicted values how many where right(75%)

Recall: Of all the actual flags how many were predicted(98%) This is good because it says that the model can flag incosisntent pages with some level of accuracy.

f1_score: How the model balances between the right predictions and the predicting the actual flags.(85%)






In [ ]:
from sklearn.metrics import f1_score,precision_score,recall_score

f1 = f1_score(y_test,test_results["model_predictions"])
precision = precision_score(y_test,test_results["model_predictions"])
recall = recall_score(y_test,test_results["model_predictions"])


print(f"f1: {f1}")
print(f"precision: {precision}")
print(f"recall: {recall}")

f1: 0.8559790514983998
precision: 0.7598140495867769
recall: 0.9800133244503664


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.